In [1]:
import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import mss
import numpy as np
import math
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp

from detectors import DETECTOR

import yaml
import time

from collections import deque

import joblib
import numpy as np
import pandas as pd


In [2]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

In [3]:
historique_predictions = deque(maxlen=120)

config = load_config("config/detector/clip_base.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/clip.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done!')

model.eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)), # Taille d'entrée de votre modèle
    transforms.ToTensor(),
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erreur : Impossible d'ouvrir la webcam.")
    exit()

print("Appuyez sur 'q' pour quitter le flux vidéo.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    start_time = time.perf_counter()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    for (x, y, w, h) in faces:
        margin_x = int(w * 0.3)
        margin_y = int(h * 0.3)
        
        x1 = max(0, x - margin_x)
        y1 = max(0, y - margin_y)
        x2 = min(frame.shape[1], x + w + margin_x)
        y2 = min(frame.shape[0], y + h + margin_y)
        
        face_img = frame[y1:y2, x1:x2]
        
        face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        
        pil_image = Image.fromarray(face_rgb)
        
        input_tensor = preprocess(pil_image).unsqueeze(0).to(device)

        with torch.no_grad():
            data_dict = {'image': input_tensor}

            output = model(data_dict, inference=True)

            probability = output["prob"].item()


        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        processing_time = end_time - start_time
        processing_time_ms = processing_time * 1000
        
        fps = 1 / processing_time if processing_time > 0 else 0

        historique_predictions.append(probability)
        
        smoothed_prob = sum(historique_predictions) / len(historique_predictions)

        label = "FAKE" if smoothed_prob > 0.98 else "REAL"
        color = (0, 0, 255) if label == "FAKE" else (0, 255, 0)
        text = f"{label}: {smoothed_prob:.2f} (Brut: {probability:.2f})"

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

        stats_text = f"Temps: {processing_time_ms:.1f} ms | FPS: {fps:.1f}"
        cv2.putText(frame, stats_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow('Deepfake Detector - POC', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Appareil utilisé : cuda
===> Load checkpoint done!
Appuyez sur 'q' pour quitter le flux vidéo.


QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot fi

In [ ]:
config = load_config("config/detector/clip_base.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/clip.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done!')

model.eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

sct = mss.mss()
zone_capture = {"top": 400, "left": 400, "width": 800, "height": 600}

print("Appuyez sur 'q' pour quitter le flux vidéo.")


active_trackers = [] 

while True:
    start_time = time.perf_counter()
    
    sct_img = sct.grab(zone_capture)
    frame = cv2.cvtColor(np.array(sct_img), cv2.COLOR_BGRA2BGR)
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    current_frame_trackers = []

    for face_i, (x, y, w, h) in enumerate(faces):
        cx = x + w // 2
        cy = y + h // 2
        
        best_tracker_idx = -1
        min_dist = float('inf')
        
        for i, tracker in enumerate(active_trackers):
            tcx, tcy = tracker['center']
            dist = math.hypot(cx - tcx, cy - tcy)
            if dist < min_dist and dist < min(w, h):
                min_dist = dist
                best_tracker_idx = i
                
        if best_tracker_idx != -1:
            matched_tracker = active_trackers.pop(best_tracker_idx)
            current_history = matched_tracker['history']
        else:
            current_history = deque(maxlen=60)
            
        current_frame_trackers.append({'center': (cx, cy), 'history': current_history})
        
        margin_x = int(w * 0.3)
        margin_y = int(h * 0.3)
        
        x1 = max(0, x - margin_x)
        y1 = max(0, y - margin_y)
        x2 = min(frame.shape[1], x + w + margin_x)
        y2 = min(frame.shape[0], y + h + margin_y)
        
        face_img = frame[y1:y2, x1:x2]
        face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(face_rgb)
        
        input_tensor = preprocess(pil_image).unsqueeze(0).to(device)

        with torch.inference_mode():
            data_dict = {'image': input_tensor}
            output = model(data_dict, inference=True)
            probability = output["prob"].item()
        
        current_history.append(probability)
        smoothed_prob = sum(current_history) / len(current_history)

        label = "FAKE" if smoothed_prob > 0.85 else "REAL"
        color = (0, 0, 255) if label == "FAKE" else (0, 255, 0)
        text = f"{label}: {smoothed_prob:.2f} (Brut: {probability:.2f})"

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    active_trackers = current_frame_trackers

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.perf_counter()
    processing_time = end_time - start_time
    fps = 1 / processing_time if processing_time > 0 else 0

    stats_text = f"Total processing time: {processing_time * 1000:.1f} ms | FPS: {fps:.1f} | Faces: {len(faces)}"
    cv2.putText(frame, stats_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.imshow('Deepfake Detector - POC', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
models = []

config = load_config("config/detector/clip_base.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/clip.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

models.append(model)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done for clip!')

model.eval().to(device)

config = load_config("config/detector/i3d.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/i3d.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

models.append(model)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done I3D!')

model.eval().to(device)

config = load_config("config/detector/xception.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-all-ff/xception.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done xception!')

model.eval().to(device)

models.append(model)

config = load_config("config/detector/spsl.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("weights/train_on_df40-fs-ff/spsl.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model = model_class(config).to(device)

if type(model).__name__ == "EffortDetector":
    model.load_state_dict(new_weights, strict=False)
else:
    model.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done SPSL!')

model.eval().to(device)

models.append(model)

cuda_streams = [torch.cuda.Stream() for _ in range(len(models))]


meta_model = joblib.load("fusion/regression_df40.pkl")

Appareil utilisé : cuda
===> Load checkpoint done for clip!
Appareil utilisé : cuda
64
loading pretrained model from /home/antoine/DF40/DeepfakeBench_DF40/training/pretrained/I3D_8x8_R50.pth
===> Load checkpoint done I3D!
Appareil utilisé : cuda
===> Load checkpoint done xception!
Appareil utilisé : cuda
===> Load checkpoint done SPSL!


In [ ]:
active_trackers = [] 

base_options = python.BaseOptions(model_asset_path='facedetection/blaze_face_short_range.tflite')
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
face_detector = vision.FaceDetector.create_from_options(options)

sct = mss.mss()
zone_capture = {"top": 1000, "left": 400, "width": 800, "height": 600}

preprocess = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
])


while True:
    start_time = time.perf_counter()
    
    sct_img = sct.grab(zone_capture)
    frame = cv2.cvtColor(np.array(sct_img), cv2.COLOR_BGRA2BGR)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    results = face_detector.detect(mp_image)
    
    faces = []
    
    if results.detections:
        for detection in results.detections:
            # La nouvelle API utilise 'bounding_box' directement
            bbox = detection.bounding_box
            
            # Les coordonnées sont déjà en pixels (entiers) !
            x = bbox.origin_x
            y = bbox.origin_y
            w = bbox.width
            h = bbox.height
            
            # Petite sécurité habituelle si le visage dépasse le bord gauche/haut
            if x < 0: 
                w += x
                x = 0
            if y < 0: 
                h += y
                y = 0
                
            faces.append((x, y, w, h))

    current_frame_trackers = []

    SEQ_LEN = 16

    for face_i, (x, y, w, h) in enumerate(faces):
        cx = x + w // 2
        cy = y + h // 2
        
        best_tracker_idx = -1
        min_dist = float('inf')
        
        for i, tracker in enumerate(active_trackers):
            tcx, tcy = tracker['center']
            dist = math.hypot(cx - tcx, cy - tcy)
            if dist < min_dist and dist < min(w, h):
                min_dist = dist
                best_tracker_idx = i
                
        if best_tracker_idx != -1:
            matched_tracker = active_trackers.pop(best_tracker_idx)
            current_history = matched_tracker['history']
            current_frames = matched_tracker['frames'] # <--- NOUVEAU : On récupère le buffer d'images
        else:
            current_history = deque(maxlen=16)
            current_frames = deque(maxlen=SEQ_LEN)     # <--- NOUVEAU : On crée le buffer d'images
            
        # 1. Préparation de l'image (identique)
        margin_x = int(w * 0.4)
        margin_y = int(h * 0.4)
        
        x1 = max(0, x - margin_x)
        y1 = max(0, y - margin_y)
        x2 = min(frame.shape[1], x + w + margin_x)
        y2 = min(frame.shape[0], y + h + margin_y)
        
        face_img = frame[y1:y2, x1:x2]
        face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(face_rgb)
        
        # input_tensor classique (1, Canaux, H, W)
        input_tensor = preprocess(pil_image).unsqueeze(0).to(device)
        
        # --- DÉBUT DE LA LOGIQUE DU BUFFER TEMPOREL ---
        # On stocke l'image sans la dimension Batch (Canaux, H, W)
        current_frames.append(input_tensor.squeeze(0))
        
        # Si le buffer n'est pas plein (ex: frame 1, 2, 3...), on duplique la dernière image
        if len(current_frames) < SEQ_LEN:
            padded_frames = list(current_frames) + [current_frames[-1]] * (SEQ_LEN - len(current_frames))
        else:
            padded_frames = list(current_frames)
            
        # On empile les frames pour créer la dimension temporelle, puis on ajoute la dimension Batch
        # Résultat : un tenseur 5D de forme (1, SEQ_LEN, Canaux, H, W)
        input_tensor_3d = torch.stack(padded_frames, dim=0).unsqueeze(0)
        # --- FIN DE LA LOGIQUE DU BUFFER ---
        
        # On sauvegarde pour la prochaine frame
        current_frame_trackers.append({'center': (cx, cy), 'history': current_history, 'frames': current_frames})

        raw_outputs = [None] * len(models)

        # 2. LANCEMENT PARALLÈLE AVEC ROUTAGE
        with torch.inference_mode():
            for i, (model, stream) in enumerate(zip(models, cuda_streams)):
                with torch.cuda.stream(stream):
                    
                    # ROUTAGE INTELLIGENT : On vérifie si c'est le modèle I3D
                    # (Adaptez la condition selon le nom de la classe de votre modèle I3D)
                    if "I3D" in type(model).__name__:
                        data_dict = {'image': input_tensor_3d}
                    else:
                        data_dict = {'image': input_tensor}
                        
                    output = model(data_dict, inference=True)
                    raw_outputs[i] = output["prob"] 
                    
        torch.cuda.synchronize()

        # 4. LECTURE ET RÉGRESSION LOGISTIQUE
        probabilities = [tensor.item() for tensor in raw_outputs]
        
        # ⚠️ IMPORTANT : Les noms de colonnes doivent être EXACTEMENT les mêmes 
        # (et dans le même ordre) que ceux de votre DataFrame d'entraînement !
        noms_colonnes = ['clip_pred', 'i3d_pred', 'xception_pred', 'spsl_preds']
        
        # On crée un DataFrame d'une ligne avec les probabilités et les bons noms
        features_df = pd.DataFrame([probabilities], columns=noms_colonnes)
        
        # On fait la prédiction sans le Warning
        prob_ensemble = float(meta_model.predict_proba(features_df)[0][1])
        #prob_ensemble = float(meta_model.predict(features_df)[0])

        # 5. Mise à jour de la moyenne glissante
        current_history.append(prob_ensemble)
        smoothed_prob = sum(current_history) / len(current_history)
        
        label = "FAKE" if smoothed_prob > 0.8 else "REAL"
        color = (0, 0, 255) if label == "FAKE" else (0, 255, 0)
        
        # Correction de la variable ici (prob_ensemble au lieu de probability)
        text = f"{label}: {smoothed_prob:.2f} (Brut: {prob_ensemble:.2f})"

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    
    active_trackers = current_frame_trackers

    end_time = time.perf_counter()
    processing_time = end_time - start_time
    fps = 1 / processing_time if processing_time > 0 else 0

    stats_text = f"Total processing time: {processing_time * 1000:.1f} ms | FPS: {fps:.1f} | Faces: {len(faces)}"
    cv2.putText(frame, stats_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    # L'affichage de la fenêtre est maintenant TOUJOURS exécuté, même sans visage !
    cv2.imshow('Deepfake Detector - POC', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

W0000 00:00:1771943876.241532  178286 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
